<div style="background-color: #f5f5f5; padding: 20px; border-radius: 10px; border: 1px solid #e0e0e0;">
<table style="width:100%; border:none;">
  <tbody><tr style="border:none;">
    <td style="width:20%; text-align:center; border:none; vertical-align:middle;">
      <img src="https://i.ibb.co/pjbk15cs/UI-Logo.png" alt="UI Logo" width="100">
    </td>
    <td style="width:60%; text-align:center; border:none; vertical-align:middle;">
      <h1 style="margin:0; color:#333;">University of Isfahan</h1>
      <h2 style="margin:0; color:#555; font-weight:normal;">Faculty of Mathematics and Statistics</h2>
      <br/>
      <hr style="border: 1px solid #ccc; margin-top:15px; margin-bottom:15px;">
      <h3 style="margin:0; color:#0056b3;">Practical Exercise: Regularization (Ridge &amp; Lasso)</h3>
      <br/>
      <h4 style="margin:0; font-size: 1.1em; color: #333; line-height: 1.6; margin-top: 15px; display: inline-block;">
        <strong>Course:</strong> Machine Learning<br>
        <strong>Instructor:</strong> Dr. Fatemeh Mansoori<br>
        <strong>Semester:</strong> Fall 2026
      </h4>
    </td>
    <td style="width:20%; text-align:center; border:none; vertical-align:middle;">
      <img src="https://i.ibb.co/pjbk15cs/UI-Logo.png" alt="UI Logo" width="100">
    </td>
  </tr>
</tbody></table>
</div>

<div style="background-color: #f5f5f5; border: 1px solid #f5f5f5; padding: 20px; border-radius: 10px; margin-top: 25px; margin-bottom: 25px;">

  <h3 style="text-align: center; margin-top: 0; color: #0056b3;">Student Information</h3>
  <p style="text-align: center; color: #555; margin-bottom: 20px; margin-top: 0;">
    Please fill in your details below.
  </p>

  <table style="width: 60%; margin-left: auto; margin-right: auto; border: none; font-size: 1.1em; color: #333;">
    <tbody>
      <tr>
        <td style="text-align: right; padding: 8px; width: 35%;"><strong>Student Name:</strong></td>
        <td style="text-align: left; padding: 8px;"><em>Your Name</em></td> <!-- UPDATE THIS LINE -->
      </tr>
      <tr>
        <td style="text-align: right; padding: 8px;"><strong>Student ID:</strong></td>
        <td style="text-align: left; padding: 8px;"><em>Your Student ID</em></td>  <!-- UPDATE THIS LINE -->
      </tr>
    </tbody>
  </table>

</div>

# **Regularization in Machine Learning: Ridge and Lasso**

In the workshop we saw that a model that is **too complex** fits the training data very well but fails on new data (**overfitting**), and that **regularization** fixes this by adding a penalty on large weights to the cost function:

$$J(\theta) = \text{Cost}(X, y, \theta) + \lambda R(\theta)$$

* **L2 / Ridge:** $R(\theta) = \sum_{j=1}^{n} \theta_j^2$ &nbsp;→ shrinks all weights smoothly towards zero.
* **L1 / Lasso:** $R(\theta) = \sum_{j=1}^{n} |\theta_j|$ &nbsp;→ drives some weights **exactly** to zero (feature selection).

In this exercise you will reproduce these ideas yourself on **new data**:

1. **Part 1 — Overfitting:** polynomial regression without regularization.
2. **Part 2 — L2 Regularization (Ridge):** the effect of $\lambda$, and choosing the best $\lambda$.
3. **Part 3 — L1 Regularization (Lasso):** sparsity and feature selection.
4. **Part 4 — Ridge vs. Lasso on real data:** regularization paths.
5. **Part 5 — Regularization in classification:** logistic regression and the parameter $C$.
6. **Bonus — Why scaling matters.**

<div style="background-color:#fff8e1; border-left: 5px solid #ffb300; padding: 12px 16px; border-radius: 6px;">
<strong>Instructions</strong>
<ul>
  <li>Every place you need to write code is marked with <code># TODO</code> inside a <code># --- YOUR TASK ---</code> block. Replace the <code>None</code> values and the <code># TODO</code> comments with your code.</li>
  <li><strong>Do not change</strong> the helper functions or the cells marked <em>(given)</em>.</li>
  <li>Some cells contain a <strong>check</strong>. If it raises an error, fix your code before moving on.</li>
  <li>Answer the questions in the <em>Your Answer</em> boxes, in your own words, based on <strong>your own plots</strong>.</li>
  <li>Before submitting: <em>Runtime → Restart and run all</em>, and make sure every cell runs without errors.</li>
</ul>
</div>

**Grading:** Part 1 — 15 pts · Part 2 — 25 pts · Part 3 — 20 pts · Part 4 — 20 pts · Part 5 — 20 pts · Bonus — +10 pts

---

## **0. Setup** *(given)*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_circles, load_diabetes, load_breast_cancer

plt.rcParams['figure.figsize'] = [12, 6]
plt.style.use('ggplot')
np.set_printoptions(precision=3, suppress=True, linewidth=100)
warnings.filterwarnings('ignore')

### **Helper Functions** *(given)*

* `mse(model, x, y)` — mean squared error of a fitted model on 1-D inputs `x`.
* `plot_fit(model, ax, title)` — draws the true function, the training points and the model's prediction.
* `plot_coef(coef, title)` — bar plot of the coefficient magnitudes $|\theta_j|$ (same as the workshop).
* `plot_2d_separator(classifier, X, ...)` — draws the decision boundary of a 2-D classifier (same as HW2).

In [ ]:
def mse(model, x, y):
    # Mean squared error of a fitted model on 1-D inputs x
    return np.mean((model.predict(x[:, None]) - y) ** 2)


def plot_fit(model, ax, title=''):
    # True function, training data and model prediction on one subplot
    x_plot = np.linspace(0, 1, 300)
    ax.plot(x_plot, f(x_plot), 'r--', alpha=0.6, label='True function')
    ax.scatter(x_train, y_train, s=25, edgecolors='k', alpha=0.8, label='Training data')
    ax.plot(x_plot, model.predict(x_plot[:, None]), 'k', lw=2, label='Model')
    ax.set_xlim(0, 1)
    ax.set_ylim(-2, 2)
    ax.set_title(title, fontsize=10)


def plot_coef(coef, title='Model Coefficients'):
    # Bar plot of |theta_j|
    coef = np.ravel(coef)
    plt.figure(figsize=(8, 3.5))
    plt.bar(np.arange(1, len(coef) + 1), np.abs(coef), color='coral', edgecolor='k')
    plt.xlabel('Coefficient index $j$ (power of $x$)')
    plt.ylabel(r'$|\theta_j|$')
    plt.title(title, fontsize=11)
    plt.show()


def plot_2d_separator(classifier, X, fill=False, ax=None, eps=None):
    # Decision boundary of a 2-D classifier
    if eps is None:
        eps = X.std() / 2.
    x_min, x_max = X[:, 0].min() - eps, X[:, 0].max() + eps
    y_min, y_max = X[:, 1].min() - eps, X[:, 1].max() + eps
    X1, X2 = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    X_grid = np.c_[X1.ravel(), X2.ravel()]
    decision_values = classifier.decision_function(X_grid).reshape(X1.shape)
    if ax is None:
        ax = plt.gca()
    if fill:
        ax.contourf(X1, X2, decision_values, levels=[decision_values.min(), 0, decision_values.max()],
                    colors=['blue', 'red'], alpha=0.1)
    ax.contour(X1, X2, decision_values, levels=[0], colors='black', linestyles=['--'])
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(())
    ax.set_yticks(())

---

## **1. Part 1: Overfitting (15 pts)**

### **The Data** *(given)*

This time the ground truth is $f(x) = \sin(2\pi x)$, corrupted with Gaussian noise. We generate:
* a **training set** of only **20 points** (the model learns from these), and
* a **test set** of **200 points** from the same function (used only to measure how well the model generalizes).

In [ ]:
def f(x):
    return np.sin(2 * np.pi * x)

rng = np.random.RandomState(33)
x_train = np.sort(rng.rand(20))
y_train = f(x_train) + 0.2 * rng.randn(20)

rng_test = np.random.RandomState(100)
x_test = np.sort(rng_test.rand(200))
y_test = f(x_test) + 0.2 * rng_test.randn(200)

plt.figure(figsize=(8, 4.5))
x_plot = np.linspace(0, 1, 300)
plt.scatter(x_train, y_train, s=50, edgecolors='k', label='Training data (20)')
plt.scatter(x_test, y_test, s=10, alpha=0.3, label='Test data (200)')
plt.plot(x_plot, f(x_plot), 'r--', lw=2, label='True function $f(x)$')
plt.xlim(0, 1)
plt.ylim(-2, 2)
plt.legend()
plt.title('Synthetic Dataset: $\\sin(2\\pi x)$ + noise')
plt.show()

### **Task 1.1 — The Model Pipeline (3 pts)**

Complete the function `make_poly_model(degree, regressor)`. It must return a `Pipeline` with three steps:
1. `'poly'`: `PolynomialFeatures(degree=degree, include_bias=False)`
2. `'scaler'`: `StandardScaler()` — scaling is important for regularization, because the penalty treats all coefficients equally
3. `'reg'`: the `regressor` passed as an argument (it can be `LinearRegression()`, `Ridge(...)`, `Lasso(...)` or even `LogisticRegression(...)`)

You will reuse this function in every part of the exercise.

In [ ]:
def make_poly_model(degree, regressor):
    # --- YOUR TASK ---
    # TODO: create the Pipeline with the three steps 'poly', 'scaler' and 'reg'
    model = None
    # --- End of YOUR TASK ---
    return model

In [ ]:
# Check (given)
_m = make_poly_model(3, LinearRegression())
assert isinstance(_m, Pipeline), 'make_poly_model must return a Pipeline.'
assert list(_m.named_steps) == ['poly', 'scaler', 'reg'], "The steps must be named 'poly', 'scaler', 'reg' (in this order)."
assert _m.named_steps['poly'].degree == 3 and _m.named_steps['poly'].include_bias is False
print('make_poly_model is correct!')

### **Task 1.2 — Polynomial Regression without Regularization (5 pts)**

For each degree in `degrees`:
1. Create a model with `make_poly_model(d, LinearRegression())`.
2. Fit it on the training data. ⚠️ `sklearn` expects a 2-D input, so use `x_train[:, None]` (shape `(20, 1)`).
3. Append the **training MSE** and the **test MSE** (use the `mse` helper) to `train_mse_deg` and `test_mse_deg`.

The plotting code is already written.

In [ ]:
degrees = [1, 3, 5, 9, 12, 15]
train_mse_deg, test_mse_deg = [], []
models_deg = {}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

for i, d in enumerate(degrees):
    # --- YOUR TASK ---
    # TODO: create the unregularized model of degree d
    model = None
    # TODO: fit the model on (x_train[:, None], y_train)

    # TODO: append the training MSE and the test MSE to the two lists
    # --- End of YOUR TASK ---

    models_deg[d] = model
    plot_fit(model, axes[i], f'Degree {d} | train MSE={train_mse_deg[-1]:.3f} | test MSE={test_mse_deg[-1]:.3f}')

plt.suptitle('Polynomial Regression without Regularization', fontsize=14)
plt.tight_layout()
plt.show()

### **Task 1.3 — Training vs. Test Error (4 pts)**

Plot `train_mse_deg` and `test_mse_deg` against `degrees` on the same figure (with markers). Use a **logarithmic y-axis**, and add axis labels, a title and a legend.

In [ ]:
plt.figure(figsize=(9, 4.5))

# --- YOUR TASK ---
# TODO: plot training MSE vs degree
# TODO: plot test MSE vs degree
# TODO: log y-axis, labels, title, legend
# --- End of YOUR TASK ---

plt.show()

### **Task 1.4 — Exploding Coefficients (3 pts)**

Get the coefficients of the **degree-15** model (`models_deg[15].named_steps['reg'].coef_`), plot them with `plot_coef`, and print the largest absolute coefficient.

In [ ]:
# --- YOUR TASK ---
# TODO: get the coefficients of the degree-15 model
coef_unreg = None

# TODO: plot them with plot_coef

# TODO: print the largest absolute coefficient (np.abs(...).max())
# --- End of YOUR TASK ---

### **Questions — Part 1**

**Question 1.1:** Which degree gives the lowest **test** error? Describe what happens to the **training** error and to the **test** error as the degree increases from 1 to 15. Which degrees underfit, and which overfit?

**Your Answer 1.1:**
> *[Write your answer here]*

<br/>

**Question 1.2:** Look at the coefficients of the degree-15 model. Why do **very large coefficients** produce a curve that oscillates wildly between the training points? How does this observation motivate regularization?

**Your Answer 1.2:**
> *[Write your answer here]*

<br/>

---

## **2. Part 2: L2 Regularization — Ridge (25 pts)**

$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} \left(h_\theta(x^{(i)}) - y^{(i)}\right)^2 + \frac{\lambda}{2m} \sum_{j=1}^{n} \theta_j^2$$

In `sklearn`, the regularization strength $\lambda$ is called **`alpha`**: `Ridge(alpha=lmbda)`.

### **Task 2.1 — The Effect of $\lambda$ (6 pts)**

Fix the degree to **15** (the model that overfit in Part 1) and try the $\lambda$ values in `lambdas_grid`. For each $\lambda$:
1. Create and fit `make_poly_model(15, Ridge(alpha=lmbda))`.
2. Compute the test MSE (`test_err`).
3. Compute the **L2 norm of the coefficients** `coef_norm` $= \|\theta\|_2$ (use `np.linalg.norm`) — a measure of how large the weights are.

In [ ]:
lambdas_grid = [1e-10, 1e-6, 1e-3, 1e-1, 1, 10]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

for i, lmbda in enumerate(lambdas_grid):
    # --- YOUR TASK ---
    # TODO: create and fit the degree-15 Ridge model with alpha=lmbda
    model = None

    # TODO: compute the test MSE
    test_err = None
    # TODO: compute the L2 norm of the coefficients
    coef_norm = None
    # --- End of YOUR TASK ---

    plot_fit(model, axes[i], f'λ = {lmbda:g} | test MSE = {test_err:.3f} | ||θ|| = {coef_norm:.1f}')

plt.suptitle('Effect of λ on Ridge Regression (Degree 15)', fontsize=14)
plt.tight_layout()
plt.show()

### **Task 2.2 — Ridge across Degrees (5 pts)**

Now do the opposite: fix $\lambda = 10^{-3}$ and repeat the degree loop of Part 1 with `Ridge` instead of `LinearRegression`. Store the test MSEs in `test_mse_ridge_deg`. The comparison plot is given.

In [ ]:
lmbda_fixed = 1e-3
test_mse_ridge_deg = []

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

for i, d in enumerate(degrees):
    # --- YOUR TASK ---
    # TODO: create and fit a Ridge model of degree d with alpha=lmbda_fixed
    model = None

    # TODO: append its test MSE to test_mse_ridge_deg
    # --- End of YOUR TASK ---

    plot_fit(model, axes[i], f'Degree {d} | test MSE = {test_mse_ridge_deg[-1]:.3f}')

plt.suptitle(f'Ridge Regression across Degrees (fixed λ = {lmbda_fixed})', fontsize=14)
plt.tight_layout()
plt.show()

# Comparison with Part 1 (given)
plt.figure(figsize=(9, 4.5))
plt.plot(degrees, test_mse_deg, 'o-', label='No regularization')
plt.plot(degrees, test_mse_ridge_deg, 's-', label=f'Ridge (λ = {lmbda_fixed})')
plt.yscale('log')
plt.xlabel('Polynomial degree')
plt.ylabel('Test MSE')
plt.title('Test error vs. degree: with and without regularization')
plt.legend()
plt.show()

### **Task 2.3 — Choosing the Best $\lambda$ (8 pts)**

Fix the degree to 15 and try 40 values of $\lambda$ between $10^{-10}$ and $10^{3}$. For each value, fit a Ridge model and append the **training MSE** and **test MSE** to the lists. Then:
* find `best_lambda_ridge`, the $\lambda$ with the **smallest test MSE** (hint: `np.argmin`),
* plot both curves against $\lambda$ with a **log x-axis** and a **log y-axis**, and mark `best_lambda_ridge` with a vertical dashed line (`plt.axvline`).

*(Note: in a real project we would choose $\lambda$ with a separate validation set or cross-validation, not the test set. Here the test set plays the role of the validation set to keep things simple.)*

In [ ]:
lambdas = np.logspace(-10, 3, 40)
train_mse_ridge, test_mse_ridge = [], []

# --- YOUR TASK ---
# TODO: loop over lambdas, fit a degree-15 Ridge model, append the training MSE and test MSE


# TODO: the lambda with the smallest test MSE
best_lambda_ridge = None

# TODO: plot training MSE and test MSE vs lambdas (log x and log y),
#       a vertical dashed line at best_lambda_ridge, labels, title, legend
# --- End of YOUR TASK ---

print(f'Best λ for Ridge: {best_lambda_ridge:.2e}   (test MSE = {min(test_mse_ridge):.4f})')

### **Task 2.4 — The Best Ridge Model (2 pts)**

Fit the degree-15 Ridge model with `best_lambda_ridge`, store it in `best_ridge`, and plot its coefficients with `plot_coef`. Compare with the coefficient plot of Task 1.4.

In [ ]:
# --- YOUR TASK ---
# TODO: fit the degree-15 Ridge model with best_lambda_ridge
best_ridge = None

# TODO: plot its coefficients with plot_coef
# --- End of YOUR TASK ---

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_fit(best_ridge, ax, f'Best Ridge model: degree 15, λ = {best_lambda_ridge:.1e}')
ax.legend()
plt.show()
print(f'Largest |theta_j|: without regularization = {np.abs(coef_unreg).max():.1f}, '
      f'with Ridge = {np.abs(best_ridge.named_steps["reg"].coef_).max():.2f}')

### **Questions — Part 2 (4 pts)**

**Question 2.1:** In Task 2.1, label each of the six plots as **"Overfitting"**, **"Good fit"** or **"Underfitting"**. What happens to $\|\theta\|_2$ as $\lambda$ increases?

**Your Answer 2.1:**
> *[Write your answer here]*

<br/>

**Question 2.2:** Compare the two curves of the comparison plot in Task 2.2. Why does the test error **no longer explode** for high degrees when we use Ridge? What does this tell you about choosing the degree when the model is regularized?

**Your Answer 2.2:**
> *[Write your answer here]*

<br/>

**Question 2.3:** In Task 2.3 the **training** error only goes **up** as $\lambda$ increases, while the test error first goes down and then up. Explain both behaviours.

**Your Answer 2.3:**
> *[Write your answer here]*

<br/>

---

## **3. Part 3: L1 Regularization — Lasso (20 pts)**

$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} \left(h_\theta(x^{(i)}) - y^{(i)}\right)^2 + \frac{\lambda}{m} \sum_{j=1}^{n} |\theta_j|$$

In `sklearn`: `Lasso(alpha=lmbda, max_iter=200000)`. Lasso needs many iterations to converge on these highly correlated polynomial features, so always pass a large `max_iter`.

### **Task 3.1 — The Effect of $\lambda$ on Lasso (6 pts)**

Fix the degree to 15. For each $\lambda$ in `lambdas_grid_l1`, fit a Lasso model, compute the test MSE, and count how many of the 15 coefficients are **exactly zero** (`n_zero`).

In [ ]:
lambdas_grid_l1 = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

for i, lmbda in enumerate(lambdas_grid_l1):
    # --- YOUR TASK ---
    # TODO: create and fit the degree-15 Lasso model (alpha=lmbda, max_iter=200000)
    model = None

    # TODO: compute the test MSE
    test_err = None
    # TODO: count the coefficients that are exactly 0
    n_zero = None
    # --- End of YOUR TASK ---

    plot_fit(model, axes[i], f'λ = {lmbda:g} | test MSE = {test_err:.3f} | zeros = {n_zero}/15')

plt.suptitle('Effect of λ on Lasso Regression (Degree 15)', fontsize=14)
plt.tight_layout()
plt.show()

### **Task 3.2 — Sparsity vs. $\lambda$ (6 pts)**

For 30 values of $\lambda$ between $10^{-5}$ and $1$, fit a degree-15 Lasso model and record the **test MSE** and the **number of zero coefficients**. Find `best_lambda_lasso` (smallest test MSE). The plot is given.

In [ ]:
lambdas_l1 = np.logspace(-5, 0, 30)
test_mse_lasso, n_zero_lasso = [], []

# --- YOUR TASK ---
# TODO: loop over lambdas_l1, fit a degree-15 Lasso model,
#       append its test MSE and its number of zero coefficients


# TODO: the lambda with the smallest test MSE
best_lambda_lasso = None
# --- End of YOUR TASK ---

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
axes[0].plot(lambdas_l1, test_mse_lasso, 's-', color='tab:blue')
axes[0].axvline(best_lambda_lasso, color='k', linestyle='--', label=f'Best λ = {best_lambda_lasso:.1e}')
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_xlabel('λ (alpha)'); axes[0].set_ylabel('Test MSE'); axes[0].set_title('Lasso: test error vs. λ'); axes[0].legend()
axes[1].step(lambdas_l1, n_zero_lasso, where='mid', color='tab:red', lw=2)
axes[1].set_xscale('log'); axes[1].set_ylim(0, 15.5)
axes[1].set_xlabel('λ (alpha)'); axes[1].set_ylabel('# coefficients = 0'); axes[1].set_title('Lasso: sparsity vs. λ')
plt.tight_layout()
plt.show()
print(f'Best λ for Lasso: {best_lambda_lasso:.2e}   (test MSE = {min(test_mse_lasso):.4f})')

### **Task 3.3 — Which Features Did Lasso Keep? (4 pts)**

1. Fit the degree-15 Lasso model with `best_lambda_lasso` and store it in `best_lasso`.
2. Plot its coefficients with `plot_coef`.
3. Print the names of the features with a **non-zero** coefficient. The names are given by `best_lasso.named_steps['poly'].get_feature_names_out(['x'])` (e.g. `'x'`, `'x^2'`, …).

In [ ]:
# --- YOUR TASK ---
# TODO: fit the degree-15 Lasso model with best_lambda_lasso
best_lasso = None

# TODO: plot its coefficients with plot_coef

# TODO: print the names of the features with a non-zero coefficient
# --- End of YOUR TASK ---

### **Questions — Part 3 (4 pts)**

**Question 3.1:** Compare the coefficient plot of the best **Ridge** model (Task 2.4) with the one of the best **Lasso** model (Task 3.3). What is the main difference? Which one performs feature selection, and why does the $|\theta_j|$ penalty produce exact zeros while $\theta_j^2$ does not?

**Your Answer 3.1:**
> *[Write your answer here]*

<br/>

**Question 3.2:** The polynomial features $x^{11}, x^{12}, \dots, x^{15}$ are extremely correlated with each other (on $[0, 1]$ they look almost the same). When several features carry almost the same information, how do Ridge and Lasso treat them differently?

**Your Answer 3.2:**
> *[Write your answer here]*

<br/>

---

## **4. Part 4: Ridge vs. Lasso on Real Data — Regularization Paths (20 pts)**

A **regularization path** shows the value of every coefficient as a function of $\lambda$. It is the clearest way to *see* the difference between L2 and L1.

We use the `diabetes` dataset from `sklearn`: 442 patients, **10 features** (age, sex, BMI, blood pressure and six blood serum measurements `s1`…`s6`). The target is a measure of disease progression one year later.

In [ ]:
diabetes = load_diabetes()
X_d = StandardScaler().fit_transform(diabetes.data)
y_d = diabetes.target
feature_names_d = diabetes.feature_names

print('X_d shape:', X_d.shape)
print('Features:', feature_names_d)

### **Task 4.1 — Compute the Paths (8 pts)**

For each `alpha` in `alphas_ridge`, fit `Ridge(alpha=alpha)` on `(X_d, y_d)` and store its `coef_` as one row of `coefs_ridge`.
Do the same with `Lasso(alpha=alpha, max_iter=10000)` for each `alpha` in `alphas_lasso` and store the results in `coefs_lasso`.

Both arrays must have shape `(60, 10)` — one row per alpha, one column per feature.

In [ ]:
alphas_ridge = np.logspace(-2, 5, 60)
alphas_lasso = np.logspace(-2, 2, 60)

# --- YOUR TASK ---
# TODO: fit one Ridge model per alpha in alphas_ridge and collect the coef_ arrays
coefs_ridge = None

# TODO: fit one Lasso model per alpha in alphas_lasso (max_iter=10000) and collect the coef_ arrays
coefs_lasso = None

# Hint: build a Python list in a loop and convert it with np.array(...) at the end
# --- End of YOUR TASK ---

assert coefs_ridge.shape == (60, 10) and coefs_lasso.shape == (60, 10), 'Wrong shape: expected (60, 10).'
print('✅ Paths computed.')

### **Task 4.2 — Plot the Paths (6 pts)**

Make a figure with **two subplots side by side**:
* **Left:** Ridge path — `alphas_ridge` on a log x-axis, one line per feature (`coefs_ridge[:, j]`), labeled with the feature name.
* **Right:** Lasso path — the same for `coefs_lasso`.

Add a horizontal line at 0, titles, axis labels and a legend.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- YOUR TASK ---
# TODO (left subplot, axes[0]): one line per feature for the Ridge path
# TODO (right subplot, axes[1]): one line per feature for the Lasso path
# TODO: for both: log x-axis, horizontal line at 0, labels, title, legend
# Hint: pass color=plt.cm.tab10(j) to ax.plot so that all 10 features get different colors
# --- End of YOUR TASK ---

plt.tight_layout()
plt.show()

### **Task 4.3 — Count the Selected Features (2 pts)**

Compute `n_nonzero`: an array with the **number of non-zero Lasso coefficients** for each alpha (length 60). Then the cell plots it and prints the surviving features.

In [ ]:
# --- YOUR TASK ---
# TODO: count the non-zero coefficients in each row of coefs_lasso
n_nonzero = None
# --- End of YOUR TASK ---

plt.figure(figsize=(10, 4))
plt.step(alphas_lasso, n_nonzero, where='mid', lw=2)
plt.xscale('log')
plt.xlabel('α')
plt.ylabel('# non-zero coefficients')
plt.title('Lasso: number of selected features vs. regularization strength')
plt.show()

for a_target in [1, 5, 20]:
    idx = np.argmin(np.abs(alphas_lasso - a_target))
    kept = [feature_names_d[j] for j in np.flatnonzero(coefs_lasso[idx])]
    print(f'alpha ≈ {alphas_lasso[idx]:6.2f}: {len(kept)} features kept -> {kept}')

### **Questions — Part 4 (4 pts)**

**Question 4.1:** Describe the main difference between the shape of the Ridge path and the Lasso path. Does Ridge ever set a coefficient **exactly** to zero? Does Lasso?

**Your Answer 4.1:**
> *[Write your answer here]*

<br/>

**Question 4.2:** According to the Lasso path, which **two or three features** are the most important for predicting diabetes progression (they survive the longest as $\alpha$ grows)? Why is this kind of result useful in practice (for example, for a doctor)?

**Your Answer 4.2:**
> *[Write your answer here]*

<br/>

---

## **5. Part 5: Regularization in Classification (20 pts)**

Regularization is just as important in classification, where an overfit model draws a jagged decision boundary around noisy points. In `sklearn`'s `LogisticRegression`, the strength is controlled by $C$:

$$J(\theta) = C \cdot \text{Cost}(X, y, \theta) + \frac{1}{2} \sum_{j=1}^{n} \theta_j^2, \qquad C = \frac{1}{\lambda}$$

* **Large $C$** → weak regularization → complex boundary (overfitting risk).
* **Small $C$** → strong regularization → simple boundary (underfitting risk).

In HW2 you used the *moons* dataset and only looked at the training data. This time we use the **circles** dataset and also measure the **test** accuracy.

In [ ]:
X_c, y_c = make_circles(n_samples=120, noise=0.2, factor=0.5, random_state=3)
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.4, random_state=0, stratify=y_c)

plt.figure(figsize=(7, 5))
plt.scatter(X_c_train[:, 0], X_c_train[:, 1], c=y_c_train, cmap='coolwarm', edgecolors='k', s=40, label='Train')
plt.scatter(X_c_test[:, 0], X_c_test[:, 1], c=y_c_test, cmap='coolwarm', marker='^', s=40, alpha=0.6, label='Test')
plt.title('Circles Dataset')
plt.legend()
plt.show()
print(f'{len(X_c_train)} training points, {len(X_c_test)} test points')

### **Task 5.1 — Decision Boundaries for Different $C$ (10 pts)**

For each value in `C_values`:
1. Create a model with your `make_poly_model(degree, LogisticRegression(C=C, max_iter=20000))` (degree 10). Yes — the same pipeline works for classification too!
2. Fit it on the **training** data `(X_c_train, y_c_train)`. (The input is already 2-D, so no `[:, None]` here.)
3. Append the **training accuracy** and the **test accuracy** (`model.score(...)`) to the two lists.

The plotting code is given.

In [ ]:
degree_c = 10
C_values = [1e-3, 1e-2, 1e-1, 1, 10, 100, 1e4, 1e6]
train_acc, test_acc = [], []

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.ravel()

for i, C in enumerate(C_values):
    # --- YOUR TASK ---
    # TODO: create the degree-10 logistic regression model with this C (max_iter=20000)
    model = None
    # TODO: fit it on the training data

    # TODO: append the training accuracy and the test accuracy
    # --- End of YOUR TASK ---

    ax = axes[i]
    plot_2d_separator(model, X_c, fill=True, ax=ax)
    ax.scatter(X_c_train[:, 0], X_c_train[:, 1], c=y_c_train, cmap='coolwarm', edgecolors='k', s=25)
    ax.set_title(f'C = {C:.0e} | train {train_acc[-1]:.0%} | test {test_acc[-1]:.0%}', fontsize=10)

plt.suptitle('Logistic Regression (degree 10): Effect of C on the Decision Boundary', fontsize=14)
plt.tight_layout()
plt.show()

### **Task 5.2 — Accuracy vs. $C$ (6 pts)**

Plot `train_acc` and `test_acc` against `C_values` with a **log x-axis**. Mark the $C$ with the highest test accuracy (`best_C`) with a vertical dashed line. Add labels, a title and a legend.

In [ ]:
plt.figure(figsize=(10, 5))

# --- YOUR TASK ---
# TODO: the C with the highest test accuracy
best_C = None

# TODO: plot training and test accuracy vs C_values (log x-axis)
# TODO: vertical dashed line at best_C, labels, title, legend
# --- End of YOUR TASK ---

plt.show()
print('Best C:', best_C)

### **Questions — Part 5 (4 pts)**

**Question 5.1:** Which values of $C$ underfit, which give a good fit, and which overfit? Use both the boundary plots and the accuracy curve to justify your answer.

**Your Answer 5.1:**
> *[Write your answer here]*

<br/>

**Question 5.2:** Compare this accuracy plot with the error plot of Task 2.3 (Ridge). Why do the two plots look like **mirror images** of each other along the x-axis? (Hint: think about the relation between $C$ and $\lambda$, and between accuracy and error.)

**Your Answer 5.2:**
> *[Write your answer here]*

<br/>

---

## **6. Bonus: Why Scaling Matters (+10 pts)**

The penalty $\lambda\sum_j |\theta_j|$ treats every coefficient the same. But in the breast cancer dataset (from HW2), `mean area` is in the hundreds or thousands while `mean smoothness` is around 0.1. What happens if we forget to scale?

### **Task 6.1 (5 pts)**

Train two models on the breast cancer training data, both with `LogisticRegression(penalty='l1', C=0.1, solver='liblinear', max_iter=5000)`:
* `unscaled_pipe`: a `Pipeline` with **only** the model (no scaler),
* `scaled_pipe`: a `Pipeline` with `StandardScaler` followed by the model.

For each one, print the **test accuracy**, the **number of non-zero coefficients** and the **names of the selected features**.

In [ ]:
cancer = load_breast_cancer()
feature_names_bc = cancer.feature_names
X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42, stratify=cancer.target)

In [ ]:
# --- YOUR TASK ---
# TODO: build and fit unscaled_pipe (model only)
unscaled_pipe = None

# TODO: build and fit scaled_pipe (StandardScaler + model)
scaled_pipe = None

# TODO: for both pipelines print the test accuracy, the number of non-zero coefficients
#       and the names of the selected features (feature_names_bc[coef != 0])
# --- End of YOUR TASK ---

### **Question 6.1 (5 pts)**

Compare the features selected **without** scaling to those selected **with** scaling. Which kind of features does the unscaled model prefer? Explain why: how does the scale (units) of a feature affect the size of its coefficient, and therefore how strongly it is penalized?

**Your Answer 6.1:**
> *[Write your answer here]*

<br/>

---

<div style="background-color:#e8f5e9; border-left: 5px solid #43a047; padding: 12px 16px; border-radius: 6px;">
<strong>Before you submit:</strong> Restart the kernel and run all cells. Make sure there are no errors, all plots appear, every check passes, and every question is answered.
</div>